# BASS vs. Fixed-Basis vs. MPS (Sec. VI.F)

Empirical (not just analytical parameter-counting) comparison across the
five circuit families already defined in the manuscript (Sec. VI.A): 1D
brickwork, 2D brickwork, RFIM (W=2), QAOA, and UCCSD.

**Kicked Ising has been removed entirely.** It is not a circuit family
defined anywhere in the current manuscript, and this notebook previously
used it in a section with its own naming bug. Introducing a brand-new,
previously-undefined circuit family this close to submission is out of
scope; it is a natural candidate for a follow-up "BASS vs. MPS on
volume-law circuits" paper, not this one.

MPS uses `chi = round(sqrt(k/N))`, the exact memory-matching criterion
already stated in Sec. VI.F of the manuscript
(`SimpleMPS.chi_for_k`).

**What changed from the previous version of this notebook:**
- The `"2D Supremacy (4x4)"` / `"2D Brickwork (4x4)"` name mismatch that
  made the ratio figure crash with a `KeyError` is gone: family names are
  now read once from `CIRCUIT_FAMILIES` and reused everywhere, there is no
  second hard-coded family list anymore.
- The redundant double `simulate()` call is gone.
- `N_TRIALS` is 100 by default (matched to the rest of the paper's
  evidentiary standard), with a `QUICK_TEST` toggle for a fast sanity run.
- All significance testing goes through
  `src/experimentation/runner.paired_comparison_stats` /
  `build_significance_table` -- the same functions used in
  `Fidelity_Pr.ipynb` and `SchmidtWeightedTruncation.ipynb` -- so the
  "Holm-Bonferroni corrected" label in the printed table is never wrong
  regardless of what's installed in the environment (no optional
  `statsmodels` dependency at all; Holm-Bonferroni is implemented directly
  in `runner.py`).
- Geometric-mean computations report how many trials underflowed to exact
  zero (`n_excluded_zero`) instead of silently dropping them.


In [1]:
from pathlib import Path
import sys, json
import numpy as np


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for c in (start, *start.parents):
        if (c / "src").is_dir() and (c / "requirements.txt").exists():
            return c
    raise RuntimeError("Repo root not found")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "tests" / "results"
FIG_DIR = REPO_ROOT / "tests" / "figures"
DATA_DIR.mkdir(exist_ok=True, parents=True)
FIG_DIR.mkdir(exist_ok=True, parents=True)

from src.simulation.bass_simulator import BASS
from src.simulation.simulator import FixedBasisSimulator
from src.simulation.mps_simulator import SimpleMPS
from src.experimentation.runner import (
    exact_statevector,
    fidelity_bass,
    make_brickwork,
    make_brickwork_2d,
    make_rfim,
    make_qaoa,
    make_uccsd,
    trial_stats,
    paired_comparison_stats,
    build_significance_table,
    format_significance_table,
)
from src.benchmarking.fidelity import compute_fidelity
from src.visualization.style import mplstyle
import matplotlib.pyplot as plt

mplstyle()
print("Imports OK. Repo root:", REPO_ROOT)


Imports OK. Repo root: c:\Users\karti\pyth\BASS


In [2]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
QUICK_TEST = False  # flip to True first to sanity-check the pipeline quickly

BASS_KW = dict(optimize_every=5, buffer_factor=1, use_2qubit_rotations=False)
SEED_BASE = 20260610
N_BOOT = 4000

if QUICK_TEST:
    N_TRIALS = 15
    K_VALUES = [200, 2000]
    CIRCUIT_FAMILIES = {
        "1D Brickwork": {"N": 10, "gen": lambda Nq, rng: make_brickwork(Nq, depth=6, rng=rng)},
        "2D Brickwork": {"N": 6, "gen": lambda Nq, rng: make_brickwork_2d(2, 3, depth=4, rng=rng)},
        "RFIM (W=2)": {"N": 10, "gen": lambda Nq, rng: make_rfim(Nq, rng, layers=5, W=2.0)},
        "QAOA (p=3)": {"N": 10, "gen": lambda Nq, rng: make_qaoa(Nq, rounds=3, rng=rng)},
        "UCCSD": {"N": 8, "gen": lambda Nq, rng: make_uccsd(Nq, rng, n_trotter_steps=1)},
    }
else:
    N_TRIALS = 100
    K_VALUES = [500, 2000, 5000, 10000, 20000]
    CIRCUIT_FAMILIES = {
        "1D Brickwork": {"N": 20, "gen": lambda Nq, rng: make_brickwork(Nq, depth=6, rng=rng)},
        "2D Brickwork": {"N": 20, "gen": lambda Nq, rng: make_brickwork_2d(4, 5, depth=4, rng=rng)},
        "RFIM (W=2)": {"N": 20, "gen": lambda Nq, rng: make_rfim(Nq, rng, layers=5, W=2.0)},
        "QAOA (p=3)": {"N": 16, "gen": lambda Nq, rng: make_qaoa(Nq, rounds=3, rng=rng)},
        "UCCSD": {"N": 12, "gen": lambda Nq, rng: make_uccsd(Nq, rng, n_trotter_steps=1)},
    }

NPZ_PATH = DATA_DIR / ("mps_comparison_quick.npz" if QUICK_TEST else "mps_comparison.npz")
META_PATH = NPZ_PATH.with_suffix(".meta.json")
EXPECTED_META = {
    "N_TRIALS": N_TRIALS,
    "K_VALUES": K_VALUES,
    "seed_base": SEED_BASE,
    "families": {name: spec["N"] for name, spec in CIRCUIT_FAMILIES.items()},
}
print(f"N_TRIALS={N_TRIALS}, k in {K_VALUES}")
print("Families:", {n: s["N"] for n, s in CIRCUIT_FAMILIES.items()})


N_TRIALS=100, k in [500, 2000, 5000, 10000, 20000]
Families: {'1D Brickwork': 20, '2D Brickwork': 20, 'RFIM (W=2)': 20, 'QAOA (p=3)': 16, 'UCCSD': 12}


In [ ]:
# ---------------------------------------------------------------------------
# Run (or load cached) sweep. Each simulator is instantiated and simulate()'d
# exactly once per trial -- no redundant calls.
# ---------------------------------------------------------------------------
def run_or_load():
    if NPZ_PATH.exists() and META_PATH.exists():
        stored_meta = json.loads(META_PATH.read_text())
        if stored_meta == EXPECTED_META:
            print("Loaded from cache:", NPZ_PATH)
            return dict(np.load(NPZ_PATH, allow_pickle=True))
        print("Cache meta mismatch -- re-running.")

    out = {}
    fam_names = list(CIRCUIT_FAMILIES.keys())
    for fi, fam_name in enumerate(fam_names):
        spec = CIRCUIT_FAMILIES[fam_name]
        Nq, gen = spec["N"], spec["gen"]
        for ki, k in enumerate(K_VALUES):
            chi = SimpleMPS.chi_for_k(Nq, k)
            f_fixed = np.full(N_TRIALS, np.nan)
            f_bass = np.full(N_TRIALS, np.nan)
            f_mps = np.full(N_TRIALS, np.nan)
            for t in range(N_TRIALS):
                seed = SEED_BASE + fi * 1_000_000 + ki * 10_000 + t
                rng = np.random.default_rng(seed)
                circuit = gen(Nq, rng)
                exact_sv = exact_statevector(Nq, circuit)

                fixed_sim = FixedBasisSimulator(Nq, k, verbose=False)
                fixed_state = fixed_sim.simulate(circuit)
                f_fixed[t] = compute_fidelity(fixed_state, exact_sv)

                bass_sim = BASS(Nq, k, **BASS_KW)
                bass_state = bass_sim.simulate(circuit)
                f_bass[t] = fidelity_bass(bass_sim, bass_state, exact_sv)

                mps = SimpleMPS(Nq, chi)
                mps.simulate(circuit)
                f_mps[t] = mps.fidelity(exact_sv)

            key = f"{fam_name}||{k}"
            out[f"{key}||f_fixed"] = f_fixed
            out[f"{key}||f_bass"] = f_bass
            out[f"{key}||f_mps"] = f_mps
            print(
                f"  {fam_name:14s} k={k:6d} (chi={chi:3d})  "
                f"med(F_fixed)={np.median(f_fixed):.3e}  "
                f"med(F_bass)={np.median(f_bass):.3e}  "
                f"med(F_mps)={np.median(f_mps):.3e}"
            )

    np.savez(NPZ_PATH, **out)
    META_PATH.write_text(json.dumps(EXPECTED_META, indent=2))
    print("Saved to", NPZ_PATH)
    return out


data = run_or_load()


  1D Brickwork   k=   500 (chi=  5)  med(F_fixed)=1.161e-06  med(F_bass)=3.499e-04  med(F_mps)=2.010e-01
  1D Brickwork   k=  2000 (chi= 10)  med(F_fixed)=9.486e-06  med(F_bass)=7.482e-04  med(F_mps)=5.805e-01
  1D Brickwork   k=  5000 (chi= 16)  med(F_fixed)=2.907e-05  med(F_bass)=2.910e-03  med(F_mps)=8.210e-01
  1D Brickwork   k= 10000 (chi= 22)  med(F_fixed)=1.666e-04  med(F_bass)=5.717e-03  med(F_mps)=8.878e-01
  1D Brickwork   k= 20000 (chi= 32)  med(F_fixed)=1.536e-03  med(F_bass)=1.656e-02  med(F_mps)=9.666e-01
  2D Brickwork   k=   500 (chi=  5)  med(F_fixed)=1.910e-04  med(F_bass)=2.517e-02  med(F_mps)=3.670e-02
  2D Brickwork   k=  2000 (chi= 10)  med(F_fixed)=1.958e-03  med(F_bass)=4.092e-02  med(F_mps)=8.405e-02
  2D Brickwork   k=  5000 (chi= 16)  med(F_fixed)=1.106e-02  med(F_bass)=7.669e-02  med(F_mps)=1.457e-01
  2D Brickwork   k= 10000 (chi= 22)  med(F_fixed)=2.590e-02  med(F_bass)=1.209e-01  med(F_mps)=2.347e-01
  2D Brickwork   k= 20000 (chi= 32)  med(F_fixed)=6.634

In [ ]:
# ---------------------------------------------------------------------------
# Publication-grade significance tables: BASS vs. Fixed-basis, BASS vs. MPS
# ---------------------------------------------------------------------------
records_vs_fixed, records_vs_mps = [], []
for fam_name in CIRCUIT_FAMILIES:
    for k in K_VALUES:
        key = f"{fam_name}||{k}"
        f_fixed = data[f"{key}||f_fixed"]
        f_bass = data[f"{key}||f_bass"]
        f_mps = data[f"{key}||f_mps"]
        label = f"{fam_name}, k={k}"
        records_vs_fixed.append(
            paired_comparison_stats(f_bass, f_fixed, label=label, n_boot=N_BOOT)
        )
        records_vs_mps.append(
            paired_comparison_stats(f_bass, f_mps, label=label, n_boot=N_BOOT)
        )

df_vs_fixed, meta_vs_fixed = build_significance_table(records_vs_fixed)
df_vs_mps, meta_vs_mps = build_significance_table(records_vs_mps)

print("=== BASS vs. Fixed-Basis ===")
print(format_significance_table(df_vs_fixed, meta_vs_fixed))
print()
print("=== BASS vs. MPS (chi = round(sqrt(k/N))) ===")
print(format_significance_table(df_vs_mps, meta_vs_mps))

df_vs_fixed.to_csv(DATA_DIR / "mps_comparison_vs_fixed.csv", index=False)
df_vs_mps.to_csv(DATA_DIR / "mps_comparison_vs_mps.csv", index=False)
print("\nSaved CSVs to", DATA_DIR)


In [ ]:
# ---------------------------------------------------------------------------
# Figure 1: fidelity vs. k, three methods, one panel per family
# ---------------------------------------------------------------------------
fam_names = list(CIRCUIT_FAMILIES.keys())
fig, axes = plt.subplots(1, len(fam_names), figsize=(4.2 * len(fam_names), 4), sharey=True)
if len(fam_names) == 1:
    axes = [axes]

colors = {"Fixed-basis": "0.4", "BASS": "#0C5DA5", "MPS": "#845B97"}
for ax, fam_name in zip(axes, fam_names):
    series = {"Fixed-basis": ([], [], []), "BASS": ([], [], []), "MPS": ([], [], [])}
    for k in K_VALUES:
        key = f"{fam_name}||{k}"
        for label, arr_key in [("Fixed-basis", "f_fixed"), ("BASS", "f_bass"), ("MPS", "f_mps")]:
            s = trial_stats(data[f"{key}||{arr_key}"])
            series[label][0].append(s["geomean"])
            series[label][1].append(s["gm_ci_lo"])
            series[label][2].append(s["gm_ci_hi"])

    for label, (gm, lo, hi) in series.items():
        ax.plot(K_VALUES, gm, "o-", label=label, color=colors[label])
        ax.fill_between(K_VALUES, lo, hi, alpha=0.15, color=colors[label])

    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Sparse budget $k$ (BASS/Fixed) / matched-memory $\\chi$ (MPS)")
    ax.set_title(fam_name)

axes[0].set_ylabel("Fidelity $F$ (geometric mean, 95% CI)")
axes[0].legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_mps_comparison_fidelity.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "fig_mps_comparison_fidelity.png", dpi=200, bbox_inches="tight")
plt.show()
print("Figure saved.")


In [ ]:
# ---------------------------------------------------------------------------
# Figure 2: summary ratio bar chart at the largest tested k, across all
# families. Labels are read directly from CIRCUIT_FAMILIES -- there is no
# second, hand-typed family-name list anywhere in this notebook, which is
# what caused the previous KeyError crash.
# ---------------------------------------------------------------------------
k_summary = K_VALUES[-1]
ratios_fixed, err_fixed_lo, err_fixed_hi = [], [], []
ratios_mps, err_mps_lo, err_mps_hi = [], [], []
for fam_name in fam_names:
    row_f = df_vs_fixed.loc[df_vs_fixed["label"] == f"{fam_name}, k={k_summary}"].iloc[0]
    row_m = df_vs_mps.loc[df_vs_mps["label"] == f"{fam_name}, k={k_summary}"].iloc[0]
    ratios_fixed.append(row_f["_ratio_gm"])
    err_fixed_lo.append(row_f["_ratio_gm"] - row_f["_ratio_ci_lo"])
    err_fixed_hi.append(row_f["_ratio_ci_hi"] - row_f["_ratio_gm"])
    ratios_mps.append(row_m["_ratio_gm"])
    err_mps_lo.append(row_m["_ratio_gm"] - row_m["_ratio_ci_lo"])
    err_mps_hi.append(row_m["_ratio_ci_hi"] - row_m["_ratio_gm"])

x = np.arange(len(fam_names))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(x - width / 2, ratios_fixed, width, yerr=[err_fixed_lo, err_fixed_hi],
       capsize=3, label="BASS / Fixed-basis", color="#0C5DA5")
ax.bar(x + width / 2, ratios_mps, width, yerr=[err_mps_lo, err_mps_hi],
       capsize=3, label="BASS / MPS", color="#845B97")
ax.axhline(1.0, color="k", lw=1, ls="--")
ax.set_yscale("log")
ax.set_xticks(x); ax.set_xticklabels(fam_names, rotation=20, ha="right")
ax.set_ylabel(f"Fidelity ratio at k={k_summary} (geometric mean, 95% CI)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_mps_comparison_ratio.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "fig_mps_comparison_ratio.png", dpi=200, bbox_inches="tight")
plt.show()
print("Figure saved.")


**Manuscript note:** Sec. VI.F can now cite `df_vs_mps`
(`tests/results/mps_comparison_vs_mps.csv`) directly for empirical fidelity
ratios, win rates, and Holm-corrected significance, replacing the
"analytical parameter counting" framing with real numbers -- while keeping
the `chi > sqrt(k/N)` argument as a complementary analytical explanation of
*why* the empirical pattern holds.
